In [ ]:
import gym
import pandas
import numpy as np

import warnings
warnings.filterwarnings("ignore")

def warn(*args, **kwargs):
    return None

warnings.warn = warn

In [ ]:
# Construa um ambiente com gym.make()
env = gym.make('FrozenLake-v1') # Construa um ambiente novo

# Inicie um novo jogo com env.reset()
current_observation = env.reset() # Isto inicia um novo "episódio" e retorna a observação inicial

# A observação atual é apenas a localização atual
print(current_observation) # Observações são apenas um número

0


In [ ]:
# podemos imprimir o ambiente se quisermos vê-lo
env.render()

In [ ]:
# o espaço de ação para este ambiente inclui quatro ações distintas

print(f"nosso espaço de ação: {env.action_space}")

new_action = env.action_space.sample() # podemos amostrar ações aleatoriamente

print(f"our new action: {new_action}") # execute esta célula algumas vezes para ter uma ideia do espaço de ação

nosso espaço de ação: Discrete(4)
our new action: 1


In [ ]:
# agora agimos! faça isso com a função step

new_action = env.action_space.sample()

observação, recompensa, concluído, info = env.step(new_action)

# aqui está uma olhada no que recebemos de volta
print(f"observação: {observação}, recompensa: {recompensa}, concluído: {concluído}, informações: {info}")

env.render()

observação: 8, recompensa: 0.0, concluído: False, informações: {'prob': 0.3333333333333333}


In [ ]:
# podemos colocar esse processo em um loop for e ver como o jogo progride

current_observation = env.reset() # inicia um novo jogo

num_mov = 5 # número de movimentos

for i in range(num_mov):

 new_action = env.action_space.sample() # igual a uma nova ação

 observação, recompensa, concluído, info = env.step(new_action) # percorre a ação e obtém os resultados

 # aqui está uma olhada no que recebemos de volta
 print(f"observação: {observação}, recompensa: {recompensa}, concluído: {concluído}, informações: {info}")

 env.render()

observação: 0, recompensa: 0.0, concluído: False, informações: {'prob': 0.3333333333333333}
observação: 4, recompensa: 0.0, concluído: False, informações: {'prob': 0.3333333333333333}
observação: 4, recompensa: 0.0, concluído: False, informações: {'prob': 0.3333333333333333}
observação: 5, recompensa: 0.0, concluído: True, informações: {'prob': 0.3333333333333333, 'TimeLimit.truncated': False}
observação: 5, recompensa: 0, concluído: True, informações: {'prob': 1.0, 'TimeLimit.truncated': False}


In [ ]:
# Veja como simular um episódio inteiro
# Vamos parar de renderizá-lo todas as vezes para economizar espaço
# tente executar isso algumas vezes. Isso alguma vez vence?

current_observation = env.reset()
done = False

while not done:
    new_action = env.action_space.sample()
    new_observation, reward, done, info = env.step(new_action)
    print(f"action:{new_action} observation: {new_observation}, reward: {reward}, done: {done}, info: {info}")

action:3 observation: 0, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:1 observation: 1, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:0 observation: 0, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:3 observation: 0, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:1 observation: 1, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:3 observation: 1, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:1 observation: 0, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:0 observation: 4, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:2 observation: 8, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:2 observation: 9, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:2 observation: 10, reward: 0.0, done: False, info: {'prob': 0.3333333333333333}
action:1 observation: 9, reward: 0.0, done: False, in

# Parte 1: Colete dados

Queremos construir um ator inteligente, mas primeiro temos de recolher dados sobre quais ações são úteis.

Use o código acima como referência. Execute um agente *aleatório* em 1.000 ou mais episódios e colete dados em cada etapa.

Eu recomendo que você armazene esses dados em um dataframe do pandas. Cada linha deve ser um degrau. Seus recursos devem incluir os seguintes recursos ou similares

- `observation` a observação no início da etapa (antes de agir!)
- `action` a ação amostrada aleatoriamente
- `current_reward` a recompensa recebida após a ação ser realizada

Depois de gerar esses dados, é recomendável calcular uma coluna (como `total_reward`, que é a recompensa total para todo o episódio).

Ao final da coleta de dados, você poderá usar pandas (ou similares) para calcular a recompensa total média *por episódio* do agente aleatório. A pontuação média deve ser de 1 a 2%, o que significa que o agente raramente vence.


## Dicas

- `initial_observation = env.reset()` inicia um novo episódio e retorna a observação inicial.
- `new_observation, recompensa, done, info = env.step(new_action)` executa uma ação e retorna a seguinte observação. Você pode consultar a documentação do método step se estiver curioso sobre o que ele faz.
- `done != True` até o jogo terminar.
- estamos tentando maximizar a recompensa *por episódio*. Nosso primeiro jogo dá 0 recompensa, a menos que o agente viaje até o gol.
- `env.action_space.n` fornece o número de ações possíveis no ambiente. `env.action_space.sample()` permite que o agente faça uma amostragem aleatória de uma ação.
- `env.observation_space.n` fornece o número de estados possíveis no ambiente.

In [ ]:
env = gym.make('FrozenLake-v1')

num_episodes = 40000

life_memory = []
for i in range(num_episodes):

    # start a new episode and record all the memories
    old_observation = env.reset()
    done = False
    tot_reward = 0
    ep_memory = []
    while not done:
        new_action = env.action_space.sample()
        observation, reward, done, info = env.step(new_action)
        tot_reward += reward

        ep_memory.append({
            "observation": old_observation,
            "action": new_action,
            "reward": reward,
            "episode": i,
        })
        old_observation = observation

    # incorporate total reward
    num_steps = len(ep_memory)
    for i, ep_mem in enumerate(ep_memory):
        ep_mem["tot_reward"] = tot_reward
        ep_mem["decay_reward"] = i*tot_reward/num_steps

    life_memory.extend(ep_memory)

memory_df = pandas.DataFrame(life_memory)

In [ ]:
memory_df.describe()

,observation,action,reward,episode,tot_reward,decay_reward
count,307685.000000,307685.000000,307685.000000,307685.000000,307685.000000,307685.000000
mean,2.216042,1.496745,0.001846,20027.941343,0.024177,0.011166
std,2.989274,1.118715,0.042926,11521.025682,0.153600,0.083898
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,10094.000000,0.000000,0.000000
50%,1.000000,1.000000,0.000000,19991.000000,0.000000,0.000000
75%,4.000000,2.000000,0.000000,29982.000000,0.000000,0.000000
max,14.000000,3.000000,1.000000,39999.000000,1.000000,0.975610


In [ ]:
memory_df.shape

(307685, 6)

In [ ]:
memory_df.groupby("episode").reward.sum().mean()

0.0142

In [ ]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.svm import SVR

model = ExtraTreesRegressor(n_estimators=50)
# model = SVR()
y = 0.5*memory_df.reward + 0.1*memory_df.decay_reward + memory_df.tot_reward
x = memory_df[["observation", "action"]]
model.fit(x, y)

ExtraTreesRegressor(n_estimators=50)

In [ ]:
model = RandomForestRegressor()
y = 1*memory_df.reward + memory_df.tot_reward + .1*memory_df.decay_reward
x = memory_df[["observation", "action"]]
model.fit(x, y)

num_episodes = 500
random_per = 0

life_memory = []
for i in range(num_episodes):
    # Start a new episode and record all the memories.
    old_observation = env.reset()
    done = False
    tot_reward = 0
    ep_memory = []
    while not done:
        if np.random.rand() < random_per:
            new_action = env.action_space.sample()
        else:
            pred_in = [[old_observation,i] for i in range(4)]
            new_action = np.argmax(model.predict(pred_in))
        observation, reward, done, info = env.step(new_action)
        tot_reward += reward

        ep_memory.append({
            "observation": old_observation,
            "action": new_action,
            "reward": reward,
            "episode": i,
        })
        old_observation = observation

    # incorporate total reward
    for ep_mem in ep_memory:
        ep_mem["tot_reward"] = tot_reward

    life_memory.extend(ep_memory)

memory_df2 = pandas.DataFrame(life_memory)

# rf.fit(memory_df[["observation", "action"]], memory_df["comb_reward"])

# Score
# Much better!
memory_df2.groupby("episode").reward.sum().mean()

0.29

In [ ]:
y = .1*memory_df.reward + 1*memory_df.decay_reward + 1*memory_df.tot_reward
y

0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
307680    0.0
307681    0.0
307682    0.0
307683    0.0
307684    0.0
Length: 307685, dtype: float64